# Dataset 6 — Individual Household Electric Power Consumption (UCI)
## Etapa B — Python / Pandas

Pré-requisito: Etapa A no Orange — atenção que aqui o dataset é carregado **original** (não a amostra usada em aula), com tratamento de ausentes feito no Orange antes de amostrar. Select Columns mantendo `Global_active_power`, `Global_reactive_power`, `Voltage`, `Global_intensity` e os três `Sub_metering`; nova amostra aleatória de 10% dos dados tratados; exportar CSV. Ajuste `CAMINHO_CSV` para o arquivo exportado.

In [1]:
import pandas as pd

CAMINHO_CSV = "household_power_consumption.csv"

df = pd.read_csv(CAMINHO_CSV)
df.columns

Index(['Date', 'Global_active_power', 'Global_reactive_power', 'Voltage',
       'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3'],
      dtype='object')

### 0. Conferência de ausentes

O tratamento principal de ausentes é feito na Etapa A (Orange), mas vale reconferir aqui — se a amostra ainda tiver ausentes, isso indica que o tratamento anterior não foi completo.

In [2]:
df.isnull().sum()

,0
Date,0
Global_active_power,2603
Global_reactive_power,2603
Voltage,2603
Global_intensity,2603
Sub_metering_1,2603
Sub_metering_2,2603
Sub_metering_3,2603


### 1. Organizar atributos com nomes mais simples

In [3]:
df = df.rename(columns={
    "Global_active_power": "Pot_Ativa",
    "Global_reactive_power": "Pot_Reativa",
    "Voltage": "Tensao",
    "Global_intensity": "Corrente",
    "Sub_metering_1": "Submedicao_1",
    "Sub_metering_2": "Submedicao_2",
    "Sub_metering_3": "Submedicao_3",
})
df.head()

,Date,Pot_Ativa,Pot_Reativa,Tensao,Corrente,Submedicao_1,Submedicao_2,Submedicao_3
0,7/7/2010,0.256,0.106,242.00,1.2,0.0,0.0,1.0
1,14/5/2007,0.466,0.352,237.22,2.4,0.0,2.0,0.0
2,26/9/2007,0.758,0.194,238.66,3.2,0.0,1.0,0.0
3,19/6/2007,1.290,0.046,240.64,5.4,1.0,0.0,18.0
4,10/5/2010,0.428,0.202,242.23,1.8,0.0,2.0,1.0


### 2. Valor máximo de potência ativa

In [4]:
max_pot_ativa = df["Pot_Ativa"].max()
max_pot_ativa

10.65

### 3. Limiar de 75% do máximo e DataFrame acima do limite

In [5]:
limiar_75 = 0.75 * max_pot_ativa
df_pot_alta = df[df["Pot_Ativa"] > limiar_75]
df_pot_alta.head()

,Date,Pot_Ativa,Pot_Reativa,Tensao,Corrente,Submedicao_1,Submedicao_2,Submedicao_3
24635,21/3/2008,8.110,0.196,236.87,34.2,37.0,74.0,17.0
42771,15/1/2008,8.026,0.058,234.91,34.0,37.0,73.0,17.0
45962,25/1/2009,8.218,0.336,236.05,34.8,37.0,73.0,17.0
46828,24/11/2009,9.708,0.372,231.37,42.0,37.0,69.0,17.0
55667,2/2/2008,8.168,0.540,228.91,35.6,35.0,35.0,17.0


### 4. Quantidade e percentual de registros selecionados

In [6]:
qtd_pot_alta = len(df_pot_alta)
percentual_pot_alta = qtd_pot_alta / len(df) * 100

print(f"Registros de potência ativa elevada: {qtd_pot_alta}")
print(f"Percentual sobre o total da amostra: {percentual_pot_alta:.2f}%")

Registros de potência ativa elevada: 33
Percentual sobre o total da amostra: 0.02%


### 5. Corrente média da amostra

In [7]:
corrente_media = df["Corrente"].mean()
corrente_media

np.float64(4.637612176280848)

### 6. Segundo DataFrame: potência ativa acima de 75% do máximo E corrente acima da média

In [8]:
df_pot_corrente_alta = df[
    (df["Pot_Ativa"] > limiar_75) &
    (df["Corrente"] > corrente_media)
]

qtd_pot_corrente_alta = len(df_pot_corrente_alta)
percentual_pot_corrente_alta = qtd_pot_corrente_alta / len(df) * 100

print(f"Registros com potência ativa alta E corrente acima da média: {qtd_pot_corrente_alta}")
print(f"Percentual sobre o total da amostra: {percentual_pot_corrente_alta:.2f}%")

Registros com potência ativa alta E corrente acima da média: 33
Percentual sobre o total da amostra: 0.02%


### 7. Comparação entre os dois conjuntos

In [9]:
print(f"Só potência ativa alta: {qtd_pot_alta} registros")
print(f"Potência ativa alta E corrente acima da média: {qtd_pot_corrente_alta} registros")
print(f"Redução ao adicionar a corrente como segunda condição: {qtd_pot_alta - qtd_pot_corrente_alta} registros a menos")

Só potência ativa alta: 33 registros
Potência ativa alta E corrente acima da média: 33 registros
Redução ao adicionar a corrente como segunda condição: 0 registros a menos


**Interpretação (preencher com os valores impressos acima antes de entregar):**

Potência ativa (`P = V × I × cos φ`) e corrente estão fisicamente relacionadas, então é esperado que boa parte dos registros de potência elevada também tenha corrente acima da média — a questão é o quanto a interseção se aproxima do conjunto original.

In [10]:
from IPython.display import display, Markdown

display(Markdown(f"""
**Interpretação:**

- Se a queda de {qtd_pot_alta} para {qtd_pot_corrente_alta} for pequena, os dois critérios são quase redundantes nesta amostra — a corrente já "carrega" a mesma informação da potência ativa, e adicionar o segundo filtro pouco refina a seleção.
- Se a queda for grande, existem episódios de potência ativa alta com corrente relativamente baixa — compatível com tensão mais alta que o normal, ou fator de potência elevado sustentando P alto sem I proporcionalmente alto. Esses casos residuais é que valeriam investigação separada, já que fogem do padrão simples de "mais corrente, mais potência".
"""))


**Interpretação:**

- Se a queda de 33 para 33 for pequena, os dois critérios são quase redundantes nesta amostra — a corrente já "carrega" a mesma informação da potência ativa, e adicionar o segundo filtro pouco refina a seleção.
- Se a queda for grande, existem episódios de potência ativa alta com corrente relativamente baixa — compatível com tensão mais alta que o normal, ou fator de potência elevado sustentando P alto sem I proporcionalmente alto. Esses casos residuais é que valeriam investigação separada, já que fogem do padrão simples de "mais corrente, mais potência".
